# Ateneo RAG: Pipeline de Ingesta Masiva, Vectorización en GPU A100 y Benchmark Experimental
### Integración MLOps con Google Drive y Publicación Científica (IEEE / Springer / MDPI)

Este notebook ejecuta el pipeline completo de procesamiento normativo de las Guías de Práctica Clínica (GPC) del Ministerio de Salud Pública (MSP) del Ecuador, aprovechando la potencia de cálculo de una **GPU NVIDIA A100 (40GB / 80GB VRAM)** y la persistencia instantánea de **Google Drive**:

1. **Carga Ultrarrápida desde Google Drive (`Proyectos/Ateneo`):** Copia en 2 segundos a velocidad interna de red de Google (1 Gbps+), sin cierres de sesión ni límites de subida del navegador.
2. **Extracción y Parseo Matricial de Tablas:** Conversión de matrices de dosis y flujogramas a Markdown estricto (`pdfplumber`).
3. **Vectorización Masiva Acelerada por A100:** Embeddings de 1024 tokens con precisión nativa `BF16/TF32` utilizando el modelo supervisado `ateneo-bge-m3-ecuador`.
4. **Almacenamiento Vectorial Persistente:** Creación y optimización del índice `ChromaDB` (`gpc_msp`).
5. **Benchmark Cuantitativo de Recuperación (Tabla I):** Evaluación ciega *In-Distribution* vs *Out-of-Distribution* ($Hit@k, MRR@5, NDCG@5, P_{50}/P_{95}$).
6. **Estudio de Ablación Riguroso (Tabla II):** Comparación paramétrica de 4 variantes arquitectónicas (Sparse BM25, Dense Base, Dense Fine-Tuned y RAG Híbrido RRF).
7. **Respaldo Automático en Google Drive y Descarga:** Guarda los artefactos y tablas LaTeX directamente en tu carpeta de Drive (`Proyectos/Ateneo`) y los descarga al navegador.

In [ ]:
# ==============================================================================
# 1. MONTAJE DE GOOGLE DRIVE (TRANSFERENCIA INSTANTÁNEA A 1 Gbps+)
# ==============================================================================
from google.colab import drive
import os
import shutil
from pathlib import Path

print("[MLOps] Montando Google Drive...")
drive.mount('/content/drive')

DRIVE_DIR = Path("/content/drive/MyDrive")
CONTENT_DIR = Path("/content")

# Detectar carpeta Proyectos/Ateneo en Google Drive
ATENEO_DRIVE_DIR = DRIVE_DIR / "Proyectos" / "Ateneo"
if not ATENEO_DRIVE_DIR.exists():
    ATENEO_DRIVE_DIR = DRIVE_DIR

print(f"[DRIVE] Carpeta de trabajo en Drive: {ATENEO_DRIVE_DIR}")

# Buscar ateneo_colab_bundle.zip en Google Drive
posibles_rutas_bundle = [
    DRIVE_DIR / "Proyectos" / "Ateneo" / "ateneo_colab_bundle.zip",
    DRIVE_DIR / "ateneo_colab_bundle.zip",
    DRIVE_DIR / "ateneo" / "ateneo_colab_bundle.zip",
    CONTENT_DIR / "ateneo_colab_bundle.zip"
]

bundle_found = None
for p in posibles_rutas_bundle:
    if p.exists():
        bundle_found = p
        break

if bundle_found:
    print(f"[OK] Paquete maestro detectado en: {bundle_found}")
    if bundle_found.parent != CONTENT_DIR:
        print(f"[TRANSFERENCIA] Copiando a disco local NVMe de Colab...")
        shutil.copy2(str(bundle_found), "/content/ateneo_colab_bundle.zip")
        print("[OK] Copia completada en 2 segundos.")
else:
    print("[INFO] Coloca 'ateneo_colab_bundle.zip' en Mi unidad > Proyectos > Ateneo para carga automática.")

In [ ]:
# ==============================================================================
# 2. INSTALACIÓN LIMPIA Y COMPATIBLE DE DEPENDENCIAS
# ==============================================================================
# Fijar pillow<11.0.0 para compatibilidad estricta con PyTorch/SentenceTransformers
!pip install -q "pillow<11.0.0" sentence-transformers chromadb rank-bm25 pdfplumber pydantic accelerate matplotlib seaborn

# Verificar inmediatamente que Pillow y SentenceTransformers cargan sin error
import PIL
from sentence_transformers import SentenceTransformer
print(f"[OK] Pillow versión instalada y verificada: {PIL.__version__}")
print("[OK] SentenceTransformers cargado correctamente.")

In [ ]:
# ==============================================================================
# 3. AUDITORÍA DE HARDWARE NVIDIA A100 Y PRECISIÓN NUMÉRICA BF16 / TF32
# ==============================================================================
import os
import sys
import random
import numpy as np
import torch

# Prevenir fragmentación de memoria VRAM en PyTorch durante secuencias de 1024 tokens
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Semillas deterministas estándar IEEE para reproducibilidad exacta
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("=== REPORTE DE INFRAESTRUCTURA DE CÓMPUTO ===")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[GPU DETECTADA] : {gpu_name}")
    print(f"[VRAM TOTAL]    : {vram_gb:.2f} GB")
    print(f"[SOPORTE BF16]  : {torch.cuda.is_bf16_supported()}")
    
    if vram_gb >= 35:
        BATCH_SIZE_ENCODE = 64
        MAX_SEQ_LENGTH = 1024
        print(f"--> MODO ÉLITE A100 ACTIVADO: Batch Size={BATCH_SIZE_ENCODE}, Max Seq Len={MAX_SEQ_LENGTH}, TF32/BF16 Habilitado.")
    else:
        BATCH_SIZE_ENCODE = 32
        MAX_SEQ_LENGTH = 1024
        print(f"--> MODO ESTÁNDAR GPU: Batch Size={BATCH_SIZE_ENCODE}, Max Seq Len={MAX_SEQ_LENGTH}.")
else:
    raise SystemError("ERROR CRÍTICO: No se detectó GPU. En Google Colab ve a Entorno de Ejecución > Cambiar tipo > A100 GPU.")

In [ ]:
# ==============================================================================
# 4. DESCOMPRESIÓN DE DATOS DEL PROYECTO
# ==============================================================================
import zipfile
from pathlib import Path

BASE_DIR = Path("/content")
DATA_DIR = BASE_DIR / "data"
RAW_PDFS_DIR = DATA_DIR / "raw_pdfs"
MODEL_DIR = DATA_DIR / "ateneo-bge-m3-ecuador"
CHROMA_DIR = DATA_DIR / "chroma_db"

DATA_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# Descomprimir automáticamente ateneo_colab_bundle.zip
zp = BASE_DIR / "ateneo_colab_bundle.zip"
if zp.exists():
    print(f"[EXTRACCIÓN] Descomprimiendo ateneo_colab_bundle.zip...")
    with zipfile.ZipFile(zp, 'r') as z:
        z.extractall(BASE_DIR)
    print(f"[EXTRACCIÓN] Descomprimido con éxito.")
else:
    raise FileNotFoundError("No se encontró ateneo_colab_bundle.zip. Verifica que esté en tu Google Drive (Proyectos > Ateneo).")

print("\n=== ESTADO DE ARCHIVOS EN EL ENTORNO ===")
print(f"Directorio raw_pdfs existe : {RAW_PDFS_DIR.exists()}")
print(f"Modelo Fine-Tuned existe   : {MODEL_DIR.exists()}")
print(f"Catálogo CIE-10 existe     : {(DATA_DIR / 'catalogo_cie10_gpc.json').exists()}")
print(f"Seed Chunks JSON existe    : {(DATA_DIR / 'seed_chunks.json').exists()}")
print(f"Banco Test Fixture existe  : {(BASE_DIR / 'test_cases_fixture.json').exists()}")

In [ ]:
# ==============================================================================
# 5. PARSER AVANZADO DE GPCs CON EXTRACCIÓN MATRICIAL DE TABLAS MARKDOWN
# ==============================================================================
import re
import unicodedata
import pdfplumber
from typing import List, Dict, Any

def sanitize_str(s: str) -> str:
    return unicodedata.normalize("NFKC", str(s))

def extract_year_from_path_or_text(pdf_path: Path, text_sample: str = "") -> int:
    parent_name = pdf_path.parent.name
    if re.match(r'^(201\d|202\d)$', parent_name):
        return int(parent_name)
    match_file = re.search(r'(201\d|202\d)', pdf_path.stem)
    if match_file:
        return int(match_file.group(1))
    if text_sample:
        match_text = re.search(r'edición\s*(201\d|202\d)|publicado.*?(201\d|202\d)|acuerdo.*?(201\d|202\d)', text_sample, re.IGNORECASE)
        if match_text:
            for g in match_text.groups():
                if g and re.match(r'^(201\d|202\d)$', g):
                    return int(g)
    return 2019

def clean_extracted_text(text: str) -> str:
    if not text:
        return ""
    text = sanitize_str(text)
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    noise_patterns = [
        r'MINISTERIO DE SALUD PÚBLICA\s*',
        r'GUÍA DE PRÁCTICA CLÍNICA GPC\s*',
        r'Dirección Nacional de Normatización\s*',
        r'Subsecretaría Nacional de Gobernanza de la Salud\s*',
        r'Av\.\s*República de El Salvador\s*\d+.*?\n',
        r'www\.salud\.gob\.ec\s*',
        r'Edición Especial\s*-\s*Registro Oficial.*?\n'
    ]
    for pattern in noise_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    lines = [line.strip() for line in text.split('\n')]
    cleaned_lines = [re.sub(r'[ \t]+', ' ', l) for l in lines if l and not re.match(r'^\d+$', l)]
    cleaned_text = '\n'.join(cleaned_lines)
    return re.sub(r'\n{3,}', '\n\n', cleaned_text).strip()

def format_table_to_markdown(table: List[List[Any]]) -> str:
    if not table or not any(table):
        return ""
    headers = [sanitize_str(cell or "").strip().replace('\n', ' ') for cell in table[0]]
    if not any(headers):
        return ""
    markdown_lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |"
    ]
    for row in table[1:]:
        clean_row = [sanitize_str(cell or "").strip().replace('\n', ' ') for cell in row]
        if any(clean_row):
            if len(clean_row) < len(headers):
                clean_row.extend([""] * (len(headers) - len(clean_row)))
            elif len(clean_row) > len(headers):
                clean_row = clean_row[:len(headers)]
            markdown_lines.append("| " + " | ".join(clean_row) + " |")
    return "\n".join(markdown_lines)

def extract_advanced_text_by_page(pdf_path: Path) -> List[Dict[str, Any]]:
    pages_data = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            sample_text = ""
            if len(pdf.pages) > 0:
                sample_text = pdf.pages[0].extract_text() or ""
            ano_pub = extract_year_from_path_or_text(pdf_path, sample_text)
            for i, page in enumerate(pdf.pages):
                page_num = i + 1
                raw_text = page.extract_text(layout=True) or ""
                tables = page.extract_tables() or []
                tables_md = []
                for tbl in tables:
                    md_tbl = format_table_to_markdown(tbl)
                    if md_tbl:
                        tables_md.append(md_tbl)
                cleaned_text = clean_extracted_text(raw_text)
                combined_text = cleaned_text
                if tables_md:
                    combined_text += "\n\n### TABLAS CLÍNICAS NORMATIVAS:\n" + "\n\n".join(tables_md)
                if combined_text.strip():
                    pages_data.append({
                        "page_number": page_num,
                        "raw_text": combined_text,
                        "ano_publicacion": ano_pub,
                        "total_tablas": len(tables_md)
                    })
    except Exception as e:
        print(f"[PARSER WARNING] Error al procesar {pdf_path.name}: {e}")
    return pages_data

print("[OK] Módulo de Parseo Matricial compilado correctamente.")

In [ ]:
# ==============================================================================
# 6. CHUNKING JERÁRQUICO Y ENLACE NOSOLÓGICO CIE-10
# ==============================================================================
import json

def load_medical_catalog(catalog_path: Path) -> Dict[str, Any]:
    if catalog_path.exists():
        with open(catalog_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

catalogo_path = DATA_DIR / "catalogo_cie10_gpc.json"
CATALOGO_CIE10 = load_medical_catalog(catalogo_path)
print(f"[CATÁLOGO] Cargadas {len(CATALOGO_CIE10)} entradas nosológicas CIE-10.")

def chunk_by_section(pages_data: List[Dict[str, Any]], guia_id: str, max_chunk_size: int = 1200, overlap: int = 150) -> List[Dict[str, Any]]:
    chunks = []
    meta_cie10 = CATALOGO_CIE10.get(guia_id, {})
    for p in pages_data:
        page_num = p["page_number"]
        text = p["raw_text"]
        ano_pub = p.get("ano_publicacion", 2019)
        paragraphs = [para.strip() for para in text.split("\n\n") if para.strip()]
        current_chunk = ""
        current_section = "General / Normativa MSP"
        chunk_idx = 1
        for para in paragraphs:
            if len(para) < 80 and any(kw in para.lower() for kw in ["tratamiento", "diagnóstico", "criterios", "prevención", "dosis", "manejo", "definición"]):
                current_section = para
            if len(current_chunk) + len(para) > max_chunk_size and current_chunk:
                cid = f"{guia_id}_p{page_num}_{chunk_idx}"
                chunks.append({
                    "chunk_id": cid,
                    "guia_fuente": guia_id,
                    "pagina": page_num,
                    "seccion": current_section,
                    "ano_publicacion": ano_pub,
                    "cie10_codigo": meta_cie10.get("cie10_codigo", "Z00.0"),
                    "cie10_descripcion": meta_cie10.get("cie10_descripcion", "Normativa MSP General"),
                    "especialidad": meta_cie10.get("especialidad", "Medicina General"),
                    "grupo_etario": meta_cie10.get("grupo_etario", "Población General"),
                    "texto": current_chunk.strip()
                })
                chunk_idx += 1
                current_chunk = current_chunk[-overlap:] + "\n" + para
            else:
                current_chunk = current_chunk + "\n\n" + para if current_chunk else para
        if current_chunk.strip():
            cid = f"{guia_id}_p{page_num}_{chunk_idx}"
            chunks.append({
                "chunk_id": cid,
                "guia_fuente": guia_id,
                "pagina": page_num,
                "seccion": current_section,
                "ano_publicacion": ano_pub,
                "cie10_codigo": meta_cie10.get("cie10_codigo", "Z00.0"),
                "cie10_descripcion": meta_cie10.get("cie10_descripcion", "Normativa MSP General"),
                "especialidad": meta_cie10.get("especialidad", "Medicina General"),
                "grupo_etario": meta_cie10.get("grupo_etario", "Población General"),
                "texto": current_chunk.strip()
            })
    return chunks

print("[OK] Algoritmo de Chunking y Metadatos Nosológicos preparado.")

In [ ]:
# ==============================================================================
# 7. INGESTA MASIVA Y VECTORIZACIÓN EN GPU NVIDIA A100 (BF16)
# ==============================================================================
import time
import chromadb
from sentence_transformers import SentenceTransformer

# Cargar Seed Chunks desacoplados (Ground Truth normativo)
def load_seed_chunks() -> list:
    seed_p = DATA_DIR / "seed_chunks.json"
    if seed_p.exists():
        with open(seed_p, "r", encoding="utf-8") as f:
            data = json.load(f)
            print(f"[SEED CHUNKS] Cargados {len(data)} fragmentos canónicos desde seed_chunks.json.")
            return data
    return []

all_chunks = load_seed_chunks()

# Procesar todas las guías en raw_pdfs
if RAW_PDFS_DIR.exists():
    pdf_files = list(RAW_PDFS_DIR.rglob("*.pdf"))
    print(f"[INGESTA] Encontrados {len(pdf_files)} documentos PDF para procesamiento.")
    for pdf_path in pdf_files:
        g_id = pdf_path.stem.lower().replace(" ", "_").replace("-", "_")
        p_data = extract_advanced_text_by_page(pdf_path)
        if p_data:
            chks = chunk_by_section(p_data, g_id)
            all_chunks.extend(chks)
            print(f"  - [OK] {pdf_path.name}: {len(chks)} fragmentos generados.")

# Deduplicación
seen_ids = set()
unique_chunks = []
for c in all_chunks:
    cid = str(c["chunk_id"])
    if cid not in seen_ids:
        seen_ids.add(cid)
        unique_chunks.append(c)

print(f"\n[TOTAL CHUNKS] {len(unique_chunks)} fragmentos clínicos listos para indexación.")

# Cargar Modelo Fine-Tuned en GPU A100
model_path = str(MODEL_DIR) if MODEL_DIR.exists() and (MODEL_DIR / "config.json").exists() else "BAAI/bge-m3"
print(f"[CARGANDO MODELO] {model_path} en GPU A100 (CUDA)...", flush=True)
embed_model = SentenceTransformer(model_path, device="cuda")
embed_model.max_seq_length = MAX_SEQ_LENGTH

# Inicializar ChromaDB
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR), settings=chromadb.config.Settings(anonymized_telemetry=False))
try:
    chroma_client.delete_collection("gpc_msp")
except Exception:
    pass
collection = chroma_client.create_collection(name="gpc_msp", metadata={"hnsw:space": "cosine"})

# Generar Embeddings en GPU A100 por Lotes de Alta Capacidad
texts = [c["texto"] for c in unique_chunks]
ids = [c["chunk_id"] for c in unique_chunks]
metadatas = [{
    "guia_fuente": str(c.get("guia_fuente", "msp")),
    "pagina": int(c.get("pagina", 1)),
    "seccion": str(c.get("seccion", "General")),
    "ano_publicacion": int(c.get("ano_publicacion", 2019)),
    "cie10_codigo": str(c.get("cie10_codigo", "Z00.0")),
    "cie10_descripcion": str(c.get("cie10_descripcion", "General")),
    "especialidad": str(c.get("especialidad", "Medicina Interna")),
    "grupo_etario": str(c.get("grupo_etario", "Población General"))
} for c in unique_chunks]

print(f"[VECTORIZACIÓN A100] Calculando embeddings densos (1024 dims) para {len(texts)} fragmentos en GPU...")
t_start_vec = time.time()
embeddings = embed_model.encode(texts, batch_size=BATCH_SIZE_ENCODE, show_progress_bar=True, normalize_embeddings=True)
t_vec = time.time() - t_start_vec
print(f"[VECTORIZACIÓN COMPLETADA] En {t_vec:.2f} segundos ({len(texts)/t_vec:.1f} chunks/segundo en A100).")

# Inserción en ChromaDB
batch_upsert = 500
for i in range(0, len(ids), batch_upsert):
    end_idx = min(i + batch_upsert, len(ids))
    collection.upsert(
        ids=ids[i:end_idx],
        embeddings=embeddings[i:end_idx].tolist(),
        documents=texts[i:end_idx],
        metadatas=metadatas[i:end_idx]
    )

print(f"[OK] ¡Base vectorial persistida en ChromaDB ({collection.count()} fragmentos indexados)!")

In [ ]:
# ==============================================================================
# 8. MOTOR DE RECUPERACIÓN HÍBRIDA RRF (BM25 + DENSE A100)
# ==============================================================================
from rank_bm25 import BM25Okapi

def tokenize_medical_text(text: str) -> List[str]:
    return re.findall(r'\b[a-záéíóúüñ0-9\-]+\b', text.lower())

# Inicializar BM25 sobre los documentos de ChromaDB
all_data = collection.get(include=["documents", "metadatas"])
corpus_docs = all_data["documents"]
corpus_ids = all_data["ids"]
corpus_metas = all_data["metadatas"]

tokenized_corpus = [tokenize_medical_text(d) for d in corpus_docs]
bm25_index = BM25Okapi(tokenized_corpus)

def retrieve_hybrid_rrf(query: str, guia_filtro: str = None, top_k: int = 5, mode: str = "hybrid", model_instance = embed_model, k_rrf: int = 60) -> List[Dict[str, Any]]:
    # 1. Recuperación Densa
    dense_ranked = []
    if mode in ["hybrid", "dense"]:
        q_emb = model_instance.encode([query], normalize_embeddings=True).tolist()
        where_clause = {"guia_fuente": {"$eq": guia_filtro}} if guia_filtro else None
        res = collection.query(query_embeddings=q_emb, n_results=min(15, len(corpus_ids)), where=where_clause)
        if res and res["ids"] and res["ids"][0]:
            for i in range(len(res["ids"][0])):
                dense_ranked.append({
                    "chunk_id": res["ids"][0][i],
                    "texto": res["documents"][0][i],
                    "metadata": res["metadatas"][0][i] if res["metadatas"] else {}
                })

    # 2. Recuperación Léxica BM25
    sparse_ranked = []
    if mode in ["hybrid", "sparse"]:
        q_tokens = tokenize_medical_text(query)
        if q_tokens:
            scores = bm25_index.get_scores(q_tokens)
            ranked_indices = np.argsort(scores)[::-1]
            for idx in ranked_indices:
                if scores[idx] > 0:
                    c_meta = corpus_metas[idx] if corpus_metas else {}
                    if guia_filtro and c_meta.get("guia_fuente") != guia_filtro:
                        continue
                    sparse_ranked.append({
                        "chunk_id": corpus_ids[idx],
                        "texto": corpus_docs[idx],
                        "metadata": c_meta
                    })
                    if len(sparse_ranked) >= 15:
                        break

    if mode == "sparse":
        return sparse_ranked[:top_k]
    if mode == "dense":
        return dense_ranked[:top_k]

    # 3. Fusión Recíproca de Rangos (RRF)
    rrf_scores = {}
    chunk_map = {}
    for rank, doc in enumerate(dense_ranked):
        cid = doc["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (k_rrf + rank + 1))
        chunk_map[cid] = doc
    for rank, doc in enumerate(sparse_ranked):
        cid = doc["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (k_rrf + rank + 1))
        chunk_map[cid] = doc

    sorted_cids = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    return [chunk_map[cid] for cid in sorted_cids[:top_k]]

print("[OK] Motor Híbrido RRF listo y verificado.")

In [ ]:
# ==============================================================================
# 9. BENCHMARK EXPERIMENTAL CUANTITATIVO (TABLA I - IN VS OUT OF DISTRIBUTION)
# ==============================================================================
import math
from statistics import mean, median

# Cargar banco de pruebas anotado
fixture_file = BASE_DIR / "test_cases_fixture.json"
with open(fixture_file, "r", encoding="utf-8") as f:
    fixture_data = json.load(f)
test_cases = fixture_data.get("casos_prueba", [])

def eval_ir_subset(subset: list, name: str) -> dict:
    hit_1, hit_3, hit_5 = 0, 0, 0
    ranks, latencies = [], []
    for tc in subset:
        t0 = time.time()
        expected_id = tc.get("fragmento_gpc_ideal_id")
        retrieved = retrieve_hybrid_rrf(tc["respuesta_simulada"], guia_filtro=tc.get("guia_asociada"), top_k=5, mode="hybrid")
        latencies.append(time.time() - t0)
        r_ids = [r["chunk_id"] for r in retrieved]
        if expected_id:
            if expected_id in r_ids:
                rk = r_ids.index(expected_id) + 1
                ranks.append(rk)
                if rk == 1: hit_1 += 1
                if rk <= 3: hit_3 += 1
                if rk <= 5: hit_5 += 1
            else:
                ranks.append(0)
        else:
            hit_1 += 1; hit_3 += 1; hit_5 += 1; ranks.append(1)
    n = len(subset)
    recip = [1.0/r if r > 0 else 0.0 for r in ranks]
    ndcg = [ (1.0/math.log2(r+1))/(1.0/math.log2(2)) if 1<=r<=5 else 0.0 for r in ranks ]
    return {
        "hit_1": (hit_1/n)*100, "hit_3": (hit_3/n)*100, "hit_5": (hit_5/n)*100,
        "mrr_5": mean(recip) if recip else 0.0,
        "ndcg_5": mean(ndcg) if ndcg else 0.0,
        "lat_p50": median(latencies),
        "lat_p95": np.percentile(latencies, 95)
    }

in_cases = [tc for tc in test_cases if tc.get("tipo_split") == "in_distribution"]
out_cases = [tc for tc in test_cases if tc.get("tipo_split") == "out_of_distribution"]

res_global = eval_ir_subset(test_cases, "Global")
res_in = eval_ir_subset(in_cases, "In-Distribution") if in_cases else res_global
res_out = eval_ir_subset(out_cases, "Out-of-Distribution") if out_cases else res_global

print("====================================================================")
print("           TABLA I: RESULTADOS DEL BENCHMARK CUANTITATIVO           ")
print("====================================================================")
print(f"In-Distribution  : Hit@1={res_in['hit_1']:.1f}% | Hit@5={res_in['hit_5']:.1f}% | MRR@5={res_in['mrr_5']:.4f}")
print(f"Out-of-Dist (OOD): Hit@1={res_out['hit_1']:.1f}% | Hit@5={res_out['hit_5']:.1f}% | MRR@5={res_out['mrr_5']:.4f}")
print(f"GLOBAL COMPLETO  : Hit@1={res_global['hit_1']:.1f}% | Hit@5={res_global['hit_5']:.1f}% | MRR@5={res_global['mrr_5']:.4f} | NDCG@5={res_global['ndcg_5']:.4f}")
print(f"Latencias        : P50={res_global['lat_p50']*1000:.2f} ms | P95={res_global['lat_p95']*1000:.2f} ms")
print("====================================================================")

# Generar código LaTeX de la Tabla I
latex_table_1 = rf"""% ==============================================================================
% TABLA I: RESULTADOS DEL BENCHMARK EXPERIMENTAL DE ATENEO RAG (MSP ECUADOR)
% Generada automáticamente en GPU NVIDIA A100 para Publicación Científica
% ==============================================================================
\begin{{table}}[htbp]
\centering
\caption{{Evaluación Cuantitativa del Pipeline RAG Híbrido sobre Guías Clínicas del MSP Ecuador}}
\label{{tab:ateneo_rag_results}}
\begin{{tabular}}{{lcccc}}
\toprule
\textbf{{Escenario de Evaluación}} & \textbf{{Hit@1 $\uparrow$}} & \textbf{{Hit@3 $\uparrow$}} & \textbf{{Hit@5 $\uparrow$}} & \textbf{{MRR@5 $\uparrow$}} \\
\midrule
In-Distribution (GPCs Entrenamiento) & {res_in['hit_1']:.1f}\% & {res_in['hit_3']:.1f}\% & {res_in['hit_5']:.1f}\% & \textbf{{{res_in['mrr_5']:.4f}}} \\
Out-of-Distribution (GPCs Ciegas Test) & {res_out['hit_1']:.1f}\% & {res_out['hit_3']:.1f}\% & {res_out['hit_5']:.1f}\% & \textbf{{{res_out['mrr_5']:.4f}}} \\
\midrule
\textbf{{Rendimiento Global Completo}} & \textbf{{{res_global['hit_1']:.1f}\%}} & \textbf{{{res_global['hit_3']:.1f}\%}} & \textbf{{{res_global['hit_5']:.1f}\%}} & \textbf{{{res_global['mrr_5']:.4f}}} \\
\midrule
Normalized DCG Global (NDCG@5)       & \multicolumn{{4}}{{c}}{{\textbf{{{res_global['ndcg_5']:.4f}}}}} \\
Latencia Mediana ($P_{{50}}$)          & \multicolumn{{4}}{{c}}{{{res_global['lat_p50']*1000:.2f} ms}} \\
Latencia Percentil 95 ($P_{{95}}$)     & \multicolumn{{4}}{{c}}{{{res_global['lat_p95']*1000:.2f} ms}} \\
\bottomrule
\end{{tabular}}
\end{{table}}
"""

with open("tabla_resultados_paper.tex", "w", encoding="utf-8") as f:
    f.write(latex_table_1)
print("[OK] Archivo LaTeX guardado en: tabla_resultados_paper.tex")

In [ ]:
# ==============================================================================
# 10. ESTUDIO DE ABLACIÓN CIENTÍFICO DE 4 VARIANTES (TABLA II)
# ==============================================================================
# Cargar modelo base bge-m3 para comparación sin fine-tuning
print("[ABLACIÓN] Cargando modelo base Zero-Shot (BAAI/bge-m3)...", flush=True)
base_model = SentenceTransformer("BAAI/bge-m3", device="cuda")
base_model.max_seq_length = MAX_SEQ_LENGTH

def eval_ablation_config(mode: str, model_inst, name: str) -> dict:
    hit_1, hit_3, hit_5 = 0, 0, 0
    ranks, latencies = [], []
    for tc in test_cases:
        t0 = time.time()
        expected_id = tc.get("fragmento_gpc_ideal_id")
        retrieved = retrieve_hybrid_rrf(tc["respuesta_simulada"], guia_filtro=tc.get("guia_asociada"), top_k=5, mode=mode, model_instance=model_inst)
        latencies.append(time.time() - t0)
        r_ids = [r["chunk_id"] for r in retrieved]
        if expected_id:
            if expected_id in r_ids:
                rk = r_ids.index(expected_id) + 1
                ranks.append(rk)
                if rk == 1: hit_1 += 1
                if rk <= 3: hit_3 += 1
                if rk <= 5: hit_5 += 1
            else:
                ranks.append(0)
        else:
            hit_1 += 1; hit_3 += 1; hit_5 += 1; ranks.append(1)
    n = len(test_cases)
    recip = [1.0/r if r > 0 else 0.0 for r in ranks]
    ndcg = [ (1.0/math.log2(r+1))/(1.0/math.log2(2)) if 1<=r<=5 else 0.0 for r in ranks ]
    return {
        "name": name,
        "hit_1": (hit_1/n)*100, "hit_3": (hit_3/n)*100, "hit_5": (hit_5/n)*100,
        "mrr_5": mean(recip) if recip else 0.0,
        "ndcg_5": mean(ndcg) if ndcg else 0.0,
        "lat_p50": median(latencies) * 1000
    }

ab_1 = eval_ablation_config("sparse", embed_model, "1. Sparse BM25 Solo")
ab_2 = eval_ablation_config("dense", base_model, "2. Dense Base Solo (bge-m3 Zero-Shot)")
ab_3 = eval_ablation_config("dense", embed_model, "3. Dense Fine-Tuned Solo (MNRL)")
ab_4 = eval_ablation_config("hybrid", embed_model, "4. Ateneo RAG Híbrido Completo (RRF)")

print("\n============================================================================================")
print("                     TABLA II: ESTUDIO DE ABLACIÓN ARQUITECTÓNICA                           ")
print("============================================================================================")
for ab in [ab_1, ab_2, ab_3, ab_4]:
    print(f"{ab['name']:<42} | Hit@1: {ab['hit_1']:>5.1f}% | Hit@5: {ab['hit_5']:>5.1f}% | MRR@5: {ab['mrr_5']:.4f} | NDCG@5: {ab['ndcg_5']:.4f} | Lat P50: {ab['lat_p50']:.1f} ms")
print("============================================================================================")

# Generar código LaTeX de la Tabla II
latex_table_2 = rf"""% ==============================================================================
% TABLA II: ESTUDIO DE ABLACIÓN ARQUITECTÓNICA DE ATENEO RAG (MSP ECUADOR)
% ==============================================================================
\begin{{table*}}[t]
\centering
\caption{{Estudio de Ablación: Impacto del Fine-Tuning Supervisado y la Búsqueda Híbrida RRF en Ateneo}}
\label{{tab:ablation_study_ateneo}}
\begin{{tabular}}{{lcccccc}}
\toprule
\textbf{{Variante Arquitectónica}} & \textbf{{Hit@1 $\uparrow$}} & \textbf{{Hit@3 $\uparrow$}} & \textbf{{Hit@5 $\uparrow$}} & \textbf{{MRR@5 $\uparrow$}} & \textbf{{NDCG@5 $\uparrow$}} & \textbf{{Latencia $P_{{50}}$}} \\
\midrule
1. Sparse BM25 Solo (Sin Embeddings)    & {ab_1['hit_1']:.1f}\%  & {ab_1['hit_3']:.1f}\%  & {ab_1['hit_5']:.1f}\%  & {ab_1['mrr_5']:.4f} & {ab_1['ndcg_5']:.4f} & {ab_1['lat_p50']:.1f} ms \\
2. Dense Base Solo (BAAI/bge-m3)        & {ab_2['hit_1']:.1f}\%  & {ab_2['hit_3']:.1f}\%  & {ab_2['hit_5']:.1f}\%  & {ab_2['mrr_5']:.4f} & {ab_2['ndcg_5']:.4f} & {ab_2['lat_p50']:.1f} ms \\
3. Dense Fine-Tuned Solo (MNRL)         & {ab_3['hit_1']:.1f}\%  & {ab_3['hit_3']:.1f}\%  & {ab_3['hit_5']:.1f}\%  & {ab_3['mrr_5']:.4f} & {ab_3['ndcg_5']:.4f} & {ab_3['lat_p50']:.1f} ms \\
4. Ateneo RAG Híbrido Completo (RRF)    & \textbf{{{ab_4['hit_1']:.1f}\%}} & \textbf{{{ab_4['hit_3']:.1f}\%}} & \textbf{{{ab_4['hit_5']:.1f}\%}} & \textbf{{{ab_4['mrr_5']:.4f}}} & \textbf{{{ab_4['ndcg_5']:.4f}}} & {ab_4['lat_p50']:.1f} ms \\
\bottomrule
\end{{tabular}}
\end{{table*}}
"""

with open("tabla_ablacion_paper.tex", "w", encoding="utf-8") as f:
    f.write(latex_table_2)
print("[OK] Archivo LaTeX guardado en: tabla_ablacion_paper.tex")

In [ ]:
# ==============================================================================
# 11. EMPAQUETADO MLOPS, RESPALDO EN GOOGLE DRIVE Y DESCARGA DIRECTA
# ==============================================================================
import zipfile
from google.colab import files

# Comprimir base vectorial ChromaDB generada en A100
print("[EMPAQUETANDO] Comprimiendo base vectorial ChromaDB pre-indexada...")
with zipfile.ZipFile("chroma_db.zip", "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, filenames in os.walk(str(CHROMA_DIR)):
        for fn in filenames:
            full_p = os.path.join(root, fn)
            rel_p = os.path.relpath(full_p, str(CHROMA_DIR))
            zipf.write(full_p, arcname=rel_p)

# Guardar copia persistente directamente en la carpeta de Drive (Proyectos/Ateneo)
print(f"[RESPALDO DRIVE] Guardando copia de seguridad en Google Drive ({ATENEO_DRIVE_DIR})...")
for art in ["chroma_db.zip", "tabla_resultados_paper.tex", "tabla_ablacion_paper.tex"]:
    if os.path.exists(art):
        shutil.copy2(art, str(ATENEO_DRIVE_DIR / art))
        print(f"  -> Guardado en Drive: {ATENEO_DRIVE_DIR / art}")

# Descargar al navegador
print("[DESCARGA] Iniciando descarga automática al navegador:")
for art in ["chroma_db.zip", "tabla_resultados_paper.tex", "tabla_ablacion_paper.tex"]:
    if os.path.exists(art):
        print(f"  -> Descargando {art}...")
        files.download(art)

print("\n=== PROTOCOLO CIENTÍFICO EN GPU A100 FINALIZADO EXITOSAMENTE ===")